# 04f — Deep Learning Exploration (MLP) v2 — Google Colab

Run on Colab with GPU for speed. Steps:
1. **Runtime → Change runtime type → T4 GPU**
2. Run cell 1 (installs)
3. Run cell 2 (mount Drive)
4. Upload the following files to `My Drive/wildfires_data/`:
   - `data/processed/train_v2.csv`
   - `data/processed/test_v2.csv`
   - `data/processed/selected_features_v2.json`
   - `outputs/v2_baseline_metrics.json`
   - `models/randomforest_v2_baseline.pkl` *(optional, for ROC comparison)*

Outputs are saved back to Drive.

In [ ]:
# ── Install dependencies not in Colab by default ────────────────────────────
!pip install -q mlflow

In [ ]:
# ── Mount Google Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, roc_curve
)
import mlflow

sns.set_theme(style='whitegrid', font_scale=1.1)

# ── Paths (all relative to your Drive folder) ────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/wildfires_data')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'outputs').mkdir(exist_ok=True)
(DRIVE_ROOT / 'models').mkdir(exist_ok=True)

PROCESSED  = DRIVE_ROOT
OUTPUTS    = DRIVE_ROOT / 'outputs'
MODELS_DIR = DRIVE_ROOT / 'models'

RANDOM_STATE = 42
CV_FOLDS     = 5
EXPERIMENT   = 'wildfires-cordoba'

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

## 1. Load Data

In [ ]:
with open(PROCESSED / 'selected_features_v2.json') as f:
    feature_meta = json.load(f)

MODEL_FEATURES  = feature_meta['all_model_features']

train = pd.read_csv(PROCESSED / 'train_v2.csv')
test  = pd.read_csv(PROCESSED / 'test_v2.csv')

X_train         = train[MODEL_FEATURES].values.astype(np.float32)
y_train         = train['label'].values.astype(np.float32)
block_ids_train = train['block_id'].values

X_test = test[MODEL_FEATURES].values.astype(np.float32)
y_test = test['label'].values.astype(np.float32)

n_features = X_train.shape[1]

print(f'Train: {X_train.shape}  fire_rate={y_train.mean():.4f}  blocks={train["block_id"].nunique()}')
print(f'Test : {X_test.shape}   fire_rate={y_test.mean():.4f}   blocks={test["block_id"].nunique()}')
print(f'Features ({n_features}): {MODEL_FEATURES}')

## 2. MLP Architecture

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim: int, dropout: float):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.block(x))


class WildfireMLP(nn.Module):
    def __init__(self, n_features: int, hidden_dims: list = [256, 256, 128], dropout: float = 0.3):
        super().__init__()
        layers = [
            nn.Linear(n_features, hidden_dims[0]),
            nn.BatchNorm1d(hidden_dims[0]), nn.GELU(), nn.Dropout(dropout),
        ]
        prev_dim = hidden_dims[0]
        for dim in hidden_dims[1:]:
            if dim == prev_dim:
                layers.append(ResidualBlock(dim, dropout))
            else:
                layers += [nn.Linear(prev_dim, dim), nn.BatchNorm1d(dim),
                           nn.GELU(), nn.Dropout(dropout)]
            prev_dim = dim
        layers += [nn.Linear(prev_dim, 1), nn.Sigmoid()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


m = WildfireMLP(n_features)
print(f'Params: {sum(p.numel() for p in m.parameters()):,}')
del m

## 3. Training Utilities

In [ ]:
def make_loader(X, y, batch_size, shuffle=True):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      pin_memory=(DEVICE.type == 'cuda'))


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item() * len(yb)
    return total / len(loader.dataset)


@torch.no_grad()
def predict_proba(model, X, batch_size=4096):
    model.eval()
    loader = make_loader(X, np.zeros(len(X)), batch_size, shuffle=False)
    return np.concatenate([model(Xb.to(DEVICE)).cpu().numpy() for Xb, _ in loader])


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'roc_auc'  : roc_auc_score(y_true, y_prob),
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
    }


def fit_mlp(X_tr, y_tr, X_val, y_val, hidden_dims=(256, 256, 128),
            lr=1e-3, weight_decay=1e-4, dropout=0.3, batch_size=1024,
            max_epochs=100, patience=15, verbose=True):
    model     = WildfireMLP(n_features, list(hidden_dims), dropout).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    criterion = nn.BCELoss()
    loader    = make_loader(X_tr, y_tr, batch_size)

    best_auc, best_state, patience_cnt = -np.inf, None, 0
    history = {'train_loss': [], 'val_auc': []}

    for epoch in range(1, max_epochs + 1):
        tr_loss  = train_epoch(model, loader, optimizer, criterion)
        scheduler.step()
        val_prob = predict_proba(model, X_val)
        val_auc  = roc_auc_score(y_val, val_prob)
        history['train_loss'].append(tr_loss)
        history['val_auc'].append(val_auc)

        if val_auc > best_auc:
            best_auc     = val_auc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                if verbose:
                    print(f'  Early stop @ epoch {epoch}  best val AUC={best_auc:.4f}')
                break
        if verbose and epoch % 10 == 0:
            print(f'  Epoch {epoch:3d} | loss={tr_loss:.4f} | val AUC={val_auc:.4f}')

    model.load_state_dict(best_state)
    return model, history, best_auc


print('Training utilities defined.')

## 4. GroupKFold CV — Architecture Search

In [ ]:
GKF = GroupKFold(n_splits=CV_FOLDS)

configs = {
    'MLP-Small'  : dict(hidden_dims=(128, 128),           dropout=0.2,  lr=5e-4),
    'MLP-Medium' : dict(hidden_dims=(256, 256, 128),      dropout=0.3,  lr=1e-3),
    'MLP-Large'  : dict(hidden_dims=(512, 256, 256, 128), dropout=0.35, lr=5e-4),
}

cv_results = {}

for cfg_name, cfg_params in configs.items():
    fold_aucs = []
    print(f'\n--- {cfg_name} ---')
    for fold_i, (tr_idx, val_idx) in enumerate(
        GKF.split(X_train, y_train, groups=block_ids_train)
    ):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]
        _, _, best_auc = fit_mlp(X_tr, y_tr, X_val, y_val,
                                  max_epochs=80, patience=12, verbose=False,
                                  **cfg_params)
        fold_aucs.append(best_auc)
        print(f'  Fold {fold_i+1}: val AUC={best_auc:.4f}')

    mean_auc = np.mean(fold_aucs)
    std_auc  = np.std(fold_aucs)
    cv_results[cfg_name] = {'mean': mean_auc, 'std': std_auc}
    print(f'  >> CV AUC = {mean_auc:.4f} ± {std_auc:.4f}')

best_cfg_name = max(cv_results, key=lambda k: cv_results[k]['mean'])
best_cfg      = configs[best_cfg_name]
print(f'\nBest: {best_cfg_name}  (CV AUC={cv_results[best_cfg_name]["mean"]:.4f})')

## 5. Train Best Architecture on Full Training Set

In [ ]:
X_tr_final, X_val_final, y_tr_final, y_val_final = train_test_split(
    X_train, y_train, test_size=0.10, random_state=RANDOM_STATE, stratify=y_train
)

print(f'Training best config: {best_cfg_name}')
best_mlp, mlp_history, _ = fit_mlp(
    X_tr_final, y_tr_final, X_val_final, y_val_final,
    max_epochs=150, patience=20, verbose=True, **best_cfg
)

mlp_prob    = predict_proba(best_mlp, X_test)
mlp_metrics = compute_metrics(y_test, mlp_prob)
print(f'\nTest AUC : {mlp_metrics["roc_auc"]:.4f}')
print(f'Test F1  : {mlp_metrics["f1"]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(mlp_history['train_loss'], color='#e74c3c', lw=2)
axes[0].set(xlabel='Epoch', ylabel='BCE Loss', title='Training Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(mlp_history['val_auc'], color='#2980b9', lw=2)
best_val = max(mlp_history['val_auc'])
axes[1].axhline(best_val, ls='--', color='#27ae60', alpha=0.8, label=f'Best={best_val:.4f}')
axes[1].set(xlabel='Epoch', ylabel='Val AUC', title='Validation AUC')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle(f'MLP Training Curves ({best_cfg_name})', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUTS / 'nn_mlp_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Comparison vs Tree Ensembles

In [ ]:
tuned_path = OUTPUTS / 'v2_tuned_metrics.json'
if tuned_path.exists():
    with open(tuned_path) as f:
        tree_rows = [{'Model': r['model'], 'Category': 'Ensemble',
                      'Test AUC': r['test_roc_auc'], 'Test F1': r['test_f1']}
                     for r in json.load(f)]
    tree_label = '(tuned)'
else:
    with open(PROCESSED / 'v2_baseline_metrics.json') as f:
        tree_rows = [{'Model': r['model'], 'Category': 'Ensemble',
                      'Test AUC': r['test_roc_auc'], 'Test F1': r['test_f1']}
                     for r in json.load(f)]
    tree_label = '(baseline)'

all_rows = tree_rows + [
    {'Model': f'MLP ({best_cfg_name})', 'Category': 'Neural Network',
     'Test AUC': mlp_metrics['roc_auc'], 'Test F1': mlp_metrics['f1']},
]
comparison_df = (pd.DataFrame(all_rows).set_index('Model')
                   .sort_values('Test AUC', ascending=False))
print(f'=== MODEL COMPARISON (trees {tree_label}) ===')
print(comparison_df.to_string())
comparison_df

In [ ]:
colors = {'Ensemble': '#2980b9', 'Neural Network': '#e67e22'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in [(axes[0], 'Test AUC'), (axes[1], 'Test F1')]:
    df_s = comparison_df.sort_values(metric, ascending=True)
    df_s[metric].plot(kind='barh', ax=ax,
                      color=[colors[c] for c in df_s['Category']], edgecolor='white')
    ax.set(xlabel=metric, title=f'{metric} — All Models')
    ax.grid(axis='x', alpha=0.3)
    for i, v in enumerate(df_s[metric]):
        ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)

from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color=c, label=k) for k, c in colors.items()],
               loc='lower right', fontsize=10)
plt.suptitle('Wildfire Susceptibility — Model Comparison (v2 Spatial CV)', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUTS / 'nn_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
fpr, tpr, _ = roc_curve(y_test, mlp_prob)
ax.plot(fpr, tpr, color='#e67e22', lw=2,
        label=f'MLP ({best_cfg_name}) AUC={mlp_metrics["roc_auc"]:.4f}')

tree_colors = ['#2980b9', '#27ae60', '#8e44ad']
for row, color in zip(tree_rows, tree_colors):
    key = row['Model'].lower().replace(' ', '')
    for suffix in ['_v2_best.pkl', '_v2_baseline.pkl']:
        mf = MODELS_DIR / f'{key}{suffix}'
        if mf.exists():
            with open(mf, 'rb') as f:
                tm = pickle.load(f)
            fpr_t, tpr_t, _ = roc_curve(y_test, tm.predict_proba(X_test)[:, 1])
            ax.plot(fpr_t, tpr_t, color=color, lw=2,
                    label=f'{row["Model"]} AUC={row["Test AUC"]:.4f}')
            break

ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.4, label='Random')
ax.set(xlabel='FPR', ylabel='TPR',
       title='ROC Curves — Ensemble vs Neural Network')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUTS / 'nn_roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Gradient-based Feature Sensitivity

In [ ]:
best_mlp.eval()
X_t = torch.tensor(X_test[:2000], dtype=torch.float32, requires_grad=True).to(DEVICE)
best_mlp(X_t).sum().backward()
grad_df = pd.Series(X_t.grad.abs().mean(dim=0).cpu().detach().numpy(),
                    index=MODEL_FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
grad_df.plot(kind='barh', ax=ax, color='#e67e22')
ax.set(title='MLP Feature Sensitivity (mean |gradient|)', xlabel='Mean absolute gradient')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUTS / 'nn_mlp_gradient_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save to Drive

In [ ]:
mlp_path = MODELS_DIR / 'mlp_v2_best.pt'
torch.save({
    'model_state_dict': best_mlp.cpu().state_dict(),
    'config'          : {k: list(v) if isinstance(v, tuple) else v
                         for k, v in best_cfg.items()},
    'config_name'     : best_cfg_name,
    'n_features'      : n_features,
    'model_features'  : MODEL_FEATURES,
    'test_metrics'    : mlp_metrics,
}, mlp_path)
print(f'Saved: {mlp_path}')

nn_summary = {
    'cv_results'      : {k: {'mean': float(v['mean']), 'std': float(v['std'])}
                         for k, v in cv_results.items()},
    'best_config_name': best_cfg_name,
    'best_config'     : {k: list(v) if isinstance(v, tuple) else v
                         for k, v in best_cfg.items()},
    'test_metrics'    : {k: float(v) for k, v in mlp_metrics.items()},
}
with open(OUTPUTS / 'nn_results.json', 'w') as f:
    json.dump(nn_summary, f, indent=2)
print('Saved: nn_results.json')
print('\nAll outputs written to Drive. Download nn_results.json and mlp_v2_best.pt.')

## 9. Verification

In [ ]:
print('=== NEURAL NETWORK V2 VERIFICATION ===')
assert mlp_metrics['roc_auc'] > 0.55
print(f'  [OK] Test AUC = {mlp_metrics["roc_auc"]:.4f}')
assert cv_results[best_cfg_name]['mean'] > 0.55
print(f'  [OK] CV AUC  = {cv_results[best_cfg_name]["mean"]:.4f}')
assert mlp_path.exists()
print(f'  [OK] {mlp_path.name} saved to Drive')

best_tree_auc = max(r['Test AUC'] for r in tree_rows)
gap = best_tree_auc - mlp_metrics['roc_auc']
print(f'\n  Best tree AUC : {best_tree_auc:.4f}')
print(f'  MLP AUC       : {mlp_metrics["roc_auc"]:.4f}')
print(f'  Gap           : {gap:+.4f}', end='  ')
if abs(gap) < 0.02:
    print('(comparable)')
elif gap > 0:
    print('(ensembles lead — expected for tabular data)')
else:
    print('(MLP wins — notable result for the paper!)')